In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

In [5]:
df = pd.read_csv('Dataset-Employee_Attrition.csv')
cols_to_drop = ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
df['Attrition'] = df['Attrition'].apply(lambda x: 1 if x == 'Yes' else 0)
df['Comp_vs_Role_Avg'] = df.groupby('JobRole')['MonthlyIncome'].transform(lambda x: x / x.mean())
X = df.drop(columns=['Attrition'])
y = df['Attrition']
X_encoded = pd.get_dummies(X, drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print("--- Model Classification Report ---")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]):.4f}")
feature_importance = pd.DataFrame({
    'Feature': X_encoded.columns,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', key=abs, ascending=False)

print("\n--- Top 10 Drivers of Employee Attrition ---")
print(feature_importance.head(10))

--- Model Classification Report ---
              precision    recall  f1-score   support

           0       0.89      0.96      0.93       247
           1       0.67      0.38      0.49        47

    accuracy                           0.87       294
   macro avg       0.78      0.67      0.71       294
weighted avg       0.86      0.87      0.86       294

ROC-AUC Score: 0.8135

--- Top 10 Drivers of Employee Attrition ---
                             Feature  Coefficient
9                      MonthlyIncome     1.343963
35     JobRole_Laboratory Technician     1.211925
38         JobRole_Research Director    -0.922547
44                      OverTime_Yes     0.872774
41      JobRole_Sales Representative     0.867070
24  BusinessTravel_Travel_Frequently     0.744352
39        JobRole_Research Scientist     0.700658
23                  Comp_vs_Role_Avg    -0.695807
36                   JobRole_Manager    -0.620122
16                 TotalWorkingYears    -0.558345
